In [6]:
import csv
import re
from pathlib import Path


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

FICHIER_LOG = Path("exemple.log")
FICHIER_CSV = Path("donnees_extraites.csv")

MARQUEUR_HEARTBEAT = "Inside Heartbeat"

NOMBRE = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"


# ------------------------------------------------------------
# Extraction de Div_pair_ratio
# ------------------------------------------------------------

PATTERN_DIV_PAIR_RATIO = re.compile(
    rf"""
    ^\s*
    (?P<heure>\d{{2}}:\d{{2}}:\d{{2}}\.\d+)
    \s+
    \[Comment\]
    .*?
    Div_pair_ratio
    \s*=\s*
    (?P<div_pair_ratio>{NOMBRE})
    """,
    re.VERBOSE | re.IGNORECASE,
)


# ------------------------------------------------------------
# Extraction des lignes Python
# ------------------------------------------------------------

PATTERN_LOG_PYTHON = re.compile(
    rf"""
    ^\s*
    (?P<heure>\d{{2}}:\d{{2}}:\d{{2}}\.\d+)
    \s+
    \[\s*\[Python\]\s+
    instr1\.getsymbol\(\)\s*=\s*'(?P<symbol>[^']+)'
    \s+
    instr1\.PrevClosePrice\(\)\s*=\s*(?P<prev_close>{NOMBRE})
    \s+
    instr1\.ContractSize\(\)\s*=\s*(?P<contract_size>{NOMBRE})
    \s+
    yld\s*=\s*(?P<yld>{NOMBRE})
    """,
    re.VERBOSE,
)


def extraire_donnees(fichier_log):
    donnees_extraites = []

    heartbeat_id = 0
    inside_heartbeat = False

    # Informations du heartbeat actuellement traité
    heure_heartbeat = None
    div_pair_ratio = None
    heure_div_pair_ratio = None

    with fichier_log.open(
        mode="r",
        encoding="utf-8",
        errors="replace",
    ) as fichier:

        for numero_ligne, ligne in enumerate(fichier, start=1):

            # ------------------------------------------------
            # 1. Détection d'un nouveau heartbeat
            # ------------------------------------------------

            if MARQUEUR_HEARTBEAT in ligne:
                heartbeat_id += 1
                inside_heartbeat = True

                # Récupération de l'heure du heartbeat
                correspondance_heure = re.match(
                    r"^\s*(\d{2}:\d{2}:\d{2}\.\d+)",
                    ligne,
                )

                if correspondance_heure is not None:
                    heure_heartbeat = correspondance_heure.group(1)
                else:
                    heure_heartbeat = None

                # Important : le nouveau heartbeat ne doit pas
                # réutiliser le ratio du heartbeat précédent.
                div_pair_ratio = None
                heure_div_pair_ratio = None

                print(
                    f"Heartbeat n°{heartbeat_id} détecté "
                    f"à la ligne {numero_ligne}"
                )

                continue

            # Ignorer les lignes avant le premier heartbeat
            if not inside_heartbeat:
                continue

            # ------------------------------------------------
            # 2. Div_pair_ratio du heartbeat courant
            # ------------------------------------------------

            correspondance_ratio = PATTERN_DIV_PAIR_RATIO.search(ligne)

            if correspondance_ratio is not None:
                groupes_ratio = correspondance_ratio.groupdict()

                div_pair_ratio = float(
                    groupes_ratio["div_pair_ratio"]
                )

                heure_div_pair_ratio = groupes_ratio["heure"]

                print(
                    f"Div_pair_ratio = {div_pair_ratio} associé "
                    f"au heartbeat n°{heartbeat_id}"
                )

                continue

            # ------------------------------------------------
            # 3. Lignes Python du heartbeat courant
            # ------------------------------------------------

            correspondance_python = PATTERN_LOG_PYTHON.search(ligne)

            if correspondance_python is None:
                continue

            groupes_python = correspondance_python.groupdict()

            donnee = {
                "heartbeat_id": heartbeat_id,
                "heure_heartbeat": heure_heartbeat,
                "numero_ligne": numero_ligne,
                "heure": groupes_python["heure"],
                "symbol": groupes_python["symbol"],
                "prev_close": float(groupes_python["prev_close"]),
                "contract_size": float(
                    groupes_python["contract_size"]
                ),
                "yld": float(groupes_python["yld"]),
                "div_pair_ratio": div_pair_ratio,
                "heure_div_pair_ratio": heure_div_pair_ratio,
            }

            donnees_extraites.append(donnee)

    return donnees_extraites


def afficher_donnees(donnees):
    if not donnees:
        print("Aucune donnée correspondant au format n'a été trouvée.")
        return

    print()
    print("Données extraites")
    print("-" * 80)

    for donnee in donnees:
        print(f"Heartbeat          : {donnee['heartbeat_id']}")
        print(f"Heure heartbeat    : {donnee['heure_heartbeat']}")
        print(f"Ligne              : {donnee['numero_ligne']}")
        print(f"Heure donnée       : {donnee['heure']}")
        print(f"Symbole            : {donnee['symbol']}")
        print(f"Clôture précédente : {donnee['prev_close']}")
        print(f"Taille contrat     : {donnee['contract_size']}")
        print(f"Yield              : {donnee['yld']}")
        print(f"Div pair ratio     : {donnee['div_pair_ratio']}")
        print(f"Heure du ratio     : {donnee['heure_div_pair_ratio']}")
        print("-" * 80)


def exporter_csv(donnees, fichier_csv):
    colonnes = [
        "heartbeat_id",
        "heure_heartbeat",
        "numero_ligne",
        "heure",
        "symbol",
        "prev_close",
        "contract_size",
        "yld",
        "div_pair_ratio",
        "heure_div_pair_ratio",
    ]

    with fichier_csv.open(
        mode="w",
        newline="",
        encoding="utf-8",
    ) as fichier:

        writer = csv.DictWriter(
            fichier,
            fieldnames=colonnes,
            delimiter=";",
        )

        writer.writeheader()
        writer.writerows(donnees)

    print()
    print(f"Fichier CSV créé : {fichier_csv.resolve()}")



In [7]:

def main():
    if not FICHIER_LOG.exists():
        print(f"Erreur : le fichier {FICHIER_LOG} n'existe pas.")
        return

    donnees = extraire_donnees(FICHIER_LOG)

    afficher_donnees(donnees)

    if donnees:
        exporter_csv(donnees, FICHIER_CSV)


if __name__ == "__main__":
    main()

Heartbeat n°1 détecté à la ligne 3
Div_pair_ratio = 1.25 associé au heartbeat n°1
Heartbeat n°2 détecté à la ligne 8
Div_pair_ratio = 0.875 associé au heartbeat n°2
Heartbeat n°3 détecté à la ligne 13
Div_pair_ratio = -0.015 associé au heartbeat n°3

Données extraites
--------------------------------------------------------------------------------
Heartbeat          : 1
Heure heartbeat    : 09:10:19.000000
Ligne              : 5
Heure donnée       : 09:10:21.000000
Symbole            : EURUSD
Clôture précédente : 1.0845
Taille contrat     : 100000.0
Yield              : 2.75
Div pair ratio     : 1.25
Heure du ratio     : 09:10:20.000000
--------------------------------------------------------------------------------
Heartbeat          : 1
Heure heartbeat    : 09:10:19.000000
Ligne              : 6
Heure donnée       : 09:10:22.000000
Symbole            : AAPL
Clôture précédente : 225.42
Taille contrat     : 100.0
Yield              : -1.25
Div pair ratio     : 1.25
Heure du ratio     :